In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv('dirty_cafe_sales.csv')

In [2]:
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [4]:
data.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [5]:
#rename the columns
data.rename(columns={
    "Transaction ID": "Transaction_ID",
    "Price Per Unit": "Price_Per_Unit",
    "Total Spent": "Total_Spent",
    "Payment Method": "Payment_Method",
    "Transaction Date": "Transaction_Date" 
    }, inplace=True)
#remove TXN_ from the ID
data['Transaction_ID'] = data['Transaction_ID'].str.replace('TXN_', '', regex=False)
#remove if there still any spaces
data['Transaction_ID'] = data['Transaction_ID'].str.strip()

In [6]:
#replace all unknown values to nan
data = data.replace(['UNKNOWN', 'ERROR', 'nan'], np.nan)

#remove all unwanted rows
data.drop_duplicates(inplace=True)
data.dropna(subset=('Transaction_Date'), inplace=True)
data.dropna(subset=['Price_Per_Unit','Quantity','Total_Spent'], how='all', inplace=True)
data.dropna(subset=['Item', 'Quantity','Total_Spent'], how='all', inplace=True)
data.dropna(subset=['Item', 'Price_Per_Unit','Total_Spent'], how='all', inplace=True)
data.dropna(subset=['Item', 'Price_Per_Unit','Quantity'], how='all', inplace=True)
data.dropna(subset=['Quantity','Total_Spent'], how='all', inplace=True)

In [7]:
#fixing the types of the columns
data['Transaction_ID'] = data['Transaction_ID'].astype(int)
data['Price_Per_Unit'] = pd.to_numeric(data['Price_Per_Unit'], errors='coerce')
data['Total_Spent'] = pd.to_numeric(data['Total_Spent'], errors='coerce')
data['Quantity'] = pd.to_numeric(data['Quantity'], errors='coerce')
data['Transaction_Date'] = pd.to_datetime(data['Transaction_Date'], errors='coerce')


In [8]:
#the mode of Items
mode3 = data.loc[data['Price_Per_Unit'] == 3, 'Item'].mode()
mode4 = data.loc[data['Price_Per_Unit'] == 4, 'Item'].mode()


In [9]:
#the items for prices
prices = {
    'Cookie': 1,
    'Tea': 1.5,
    'Coffee': 2,
    'Cake': 3,
    'Juice': 3,
    'Sandwich': 4,
    'Smoothie': 4,
    'Salad': 5
}

#the prices for items
items = {
    1: 'Cookie',
    1.5: 'Tea',
    2: 'Coffee',
    3: mode3[0],
    4: mode4[0],
    5: 'Salad'
}
#getting the price per unit from the items
mask = (data['Price_Per_Unit'].isna()) & (data['Item'].notna())
data.loc[mask, 'Price_Per_Unit'] = data.loc[mask, 'Item'].map(prices)

#getting the price per unit from total / quantity (there is no need for this but i did it incase to be known)
mask2 = (data['Quantity'].notna()) & (data['Total_Spent'].notna()) & (data['Price_Per_Unit'].isna())
data.loc[mask2, 'Price_Per_Unit'] = data.loc[mask2, 'Total_Spent'] / data.loc[mask2, 'Quantity']

#getting the items from the price per unit
mask3 = (data['Item'].isna()) & (data['Price_Per_Unit'].notna())
data.loc[mask3, 'Item'] = data.loc[mask3, 'Price_Per_Unit'].map(items)

#getting the quantity from total / price per unit
mask4 = (data['Total_Spent'].notna()) & (data['Price_Per_Unit'].notna())
data.loc[mask4, 'Quantity'] = data.loc[mask4, 'Total_Spent'] / data.loc[mask4, 'Price_Per_Unit']

#getting the total from quantity * price per unit
mask5 = (data['Quantity'].notna()) & (data['Price_Per_Unit'].notna()) & (data['Total_Spent'].isna())
data.loc[mask5, 'Total_Spent'] = data.loc[mask5, 'Quantity'] * data.loc[mask5, 'Price_Per_Unit']




In [10]:
#mode for location
mode_location = data['Location'].mode()

data['Location'] = data['Location'].fillna(mode_location[0])



In [11]:
#mode for payment
mode_payment = data['Payment_Method'].mode()

data['Payment_Method'] = data['Payment_Method'].fillna(mode_payment[0])


In [12]:
data.head()

,Transaction_ID,Item,Quantity,Price_Per_Unit,Total_Spent,Payment_Method,Location,Transaction_Date
0,1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,7034554,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway,2023-04-27
4,3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [13]:
#getting round for total and price per unit
data['Total_Spent'] = round(data['Total_Spent'],2)
data['Price_Per_Unit'] = round(data['Price_Per_Unit'],2)

In [14]:
#the seasons 
date = data['Transaction_Date']
def get_season(date):
        month = date.month
        if month in [11, 12, 1]:
            return 'Winter'
        elif month in [2, 3, 4]:
            return 'Spring'
        elif month in [5, 6, 7]:
            return 'Summer'
        else:
            return 'Fall'

# # إنشاء العمود الجديد
data['Season'] = data['Transaction_Date'].apply(get_season)

In [15]:
data.groupby("Season")['Total_Spent'].sum()

Season
Fall      21277.5
Spring    21039.0
Summer    21163.0
Winter    21389.0
Name: Total_Spent, dtype: float64

In [16]:
seasonal_stats = data.groupby("Season").agg({
    "Transaction_ID": "count",
    "Total_Spent": ["sum","max","min"],
    "Transaction_Date": "count"
}).reset_index()

seasonal_stats.to_csv("seasonal_sales_summary.csv", index=False, encoding='utf-8')

In [17]:
data.to_csv('cleaned_cafe_sales.csv', index=False, encoding='utf-8')

In [18]:
Sorted_By_ID = data.sort_values(by=['Transaction_ID'])
Sorted_By_ID.to_csv('Sorted_By_ID.csv', index=False, encoding='utf-8')

In [ ]:
import matplotlib as plt
data.plot(subplots=True)

ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.